In [0]:
import os
catalog, schema = "workspace", "dm2_project"
base = f"/Volumes/{catalog}/{schema}"
ref, land = f"{base}/reference", f"{base}/landing/sales"

# reset bronze + landing so we can load the better sample
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.bronze_sales")
dbutils.fs.rm(f"{base}/chk/bronze_sales", True)
dbutils.fs.rm(f"{base}/chk/_schema/bronze_sales", True)
try:
    for f in dbutils.fs.ls(land):
        dbutils.fs.rm(f.path, True)
except Exception:
    pass

# representative sample across the FULL dataset (many months + countries)
retail = [f for f in os.listdir(ref) if f.lower().endswith(".csv")
          and "country" not in f.lower() and "test" not in f.lower()][0]
full   = spark.read.option("header", "true").csv(f"{ref}/{retail}")
sample = full.sample(False, 0.006, seed=42)
part1, part2 = sample.randomSplit([0.7, 0.3], seed=1)

os.makedirs(land, exist_ok=True)
part1.toPandas().to_csv(f"{land}/sales_part1.csv", index=False)
part2.toPandas().to_csv(f"{ref}/sales_part2_TEST.csv", index=False)

print("load rows :", part1.count())
print("test rows :", part2.count())
print("months in load   :", part1.selectExpr("substr(InvoiceDate,1,7) AS m").distinct().count())
print("countries in load:", part1.select("Country").distinct().count())

In [0]:
import shutil
base = "/Volumes/workspace/dm2_project"
shutil.copy(f"{base}/reference/sales_part2_TEST.csv",
            f"{base}/landing/sales/sales_part2.csv")
print("test file dropped into landing/sales")